### 1. Imports

In [1]:
import numpy as np
import pandas as pd 
import seaborn as sns
import json
import unidecode
from matplotlib import pyplot as plt
import ast
from flaml import AutoML
from sklearn.cluster import KMeans
import ast

c:\Users\rlope\anaconda3\envs\hc_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-12-11 11:13:52,948	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2025-12-11 11:13:53,223	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
# Evitar printar warnings
import warnings 
warnings.filterwarnings("ignore")

### 2. Load data and Overview

In [3]:
train_df = pd.read_csv('../data/train_split.csv')
test_df = pd.read_csv('../data/test_split.csv') 

# Train & test dataframes  shape
print(train_df.shape, test_df.shape)

(110000, 70) (39867, 70)


In [4]:
df_train_embeddings = pd.read_csv('../data/hc_proyecto_train_data_embeddings.csv')
df_test_embeddings = pd.read_csv('../data/hc_proyecto_test_data_embeddings.csv')


In [5]:
def parse_numpy_str(s):
    # 1. Quitamos corchetes
    s = s.strip("[]")
    # 2. Reemplazamos comas por espacios (por si acaso vienen separados por comas)
    s = s.replace(',', ' ')
    # 3. Convertimos
    arr = np.fromstring(s, sep=' ').tolist()
    return arr

# 2. Aplicarlo a tu DataFrame
df_train_embeddings['descripcion_embeddings'] = df_train_embeddings['descripcion_embeddings'].apply(parse_numpy_str)
df_test_embeddings['descripcion_embeddings'] = df_test_embeddings['descripcion_embeddings'].apply(parse_numpy_str)


In [6]:
embedding_dim = len(df_train_embeddings.iloc[0]["descripcion_embeddings"])
# Convertir la columna de embeddings en una matriz 2D
emb_matrix = np.vstack(df_train_embeddings["descripcion_embeddings"].values)
# Crear un nuevo dataframe de columnas emb_0, emb_1, ...
emb_df = pd.DataFrame(
    emb_matrix,
    columns=[f"emb_{i}" for i in range(embedding_dim)]
)

embedding_dim_2 = len(df_test_embeddings.iloc[0]["descripcion_embeddings"])
# Convertir la columna de embeddings en una matriz 2D
emb_matrix_2 = np.vstack(df_test_embeddings["descripcion_embeddings"].values)
# Crear un nuevo dataframe de columnas emb_0, emb_1, ...
emb_df_2 = pd.DataFrame(
    emb_matrix_2,
    columns=[f"emb_{i}" for i in range(embedding_dim_2)]
)


train_df = pd.concat([train_df, emb_df],axis=1)
test_df = pd.concat([test_df, emb_df_2],axis=1)


In [7]:
# First overview of train dataframe features/target info
train_df.info()
# # Look at top 5 rows
train_df.head(3)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 110000 entries, 0 to 109999
Columns: 326 entries, id to emb_255
dtypes: bool(1), float64(276), int64(3), object(46)
memory usage: 272.9+ MB


,id,ciudad,estado,estado_propiedad,dormitorios,banos,precio,anio_construccion,codigo_postal,longitud,...,emb_246,emb_247,emb_248,emb_249,emb_250,emb_251,emb_252,emb_253,emb_254,emb_255
0,1,Lakeland,FL,FOR_SALE,3.0,2.0,119900.0,NaN,33810,-81.97661,...,0.055814,0.083686,-0.002043,0.069716,0.094002,-0.034311,-0.043107,0.045225,0.055779,-0.000019
1,2,Pompano Beach,FL,FOR_SALE,3.0,2.0,290000.0,NaN,33067,-80.24621,...,0.054837,0.142771,-0.021663,0.012974,0.083022,-0.025964,-0.021053,-0.034879,0.058153,0.063659
2,3,Jacksonville,FL,FOR_SALE,3.0,3.0,599000.0,NaN,32205,-81.71293,...,0.022639,0.138726,-0.025745,0.000391,0.048933,-0.099128,0.054381,0.026659,-0.045843,0.021626


In [8]:
# Elimino las columnas id de ambos datasets
train_df.drop(columns=['id'], inplace=True)

test_ids_list = test_df['id'].values # Extraemos para luego usarlo en el submission 
test_df.drop(columns=['id'], inplace=True)

In [9]:
len(test_ids_list)

39867

In [10]:
# Train & test dataframes  shape
print(train_df.shape, test_df.shape)

(110000, 325) (39867, 325)


In [11]:
# Determino intersección en columnas entre train y test
common_cols = train_df.columns.intersection(test_df.columns)
len(common_cols)

325

In [12]:
# Cargo las feature importances de un entrenamiento previo de autogluon: 

df_feature_importances = pd.read_csv('../data/autogluon_submission5_importances.csv').sort_values(by='importance',ascending=False)
df_feature_importances


,Unnamed: 0,importance,stddev,p_value,n,p99_high,p99_low
51,descripcion,2.227178e-01,8.809284e-03,2.931019e-07,5,2.408562e-01,2.045794e-01
50,ciudad,1.921730e-01,2.428680e-03,3.060559e-09,5,1.971737e-01,1.871723e-01
49,valor_area_habitable,1.807573e-01,2.696772e-03,5.943558e-09,5,1.863100e-01,1.752046e-01
48,latitud,1.516501e-01,4.131508e-03,6.604125e-08,5,1.601569e-01,1.431433e-01
47,tipo_vivienda,1.157187e-01,1.922989e-03,9.147754e-09,5,1.196782e-01,1.117592e-01
46,longitud,1.113356e-01,4.575521e-03,3.415305e-07,5,1.207567e-01,1.019146e-01
45,interior_completo,1.068882e-01,1.791449e-03,9.464939e-09,5,1.105768e-01,1.031995e-01
44,area_habitable,1.043969e-01,2.033041e-03,1.725025e-08,5,1.085830e-01,1.002109e-01
43,tamano_lote,8.498194e-02,3.058599e-03,2.010088e-07,5,9.127964e-02,7.868425e-02
42,construccion,7.965611e-02,5.954555e-04,3.746881e-10,5,8.088216e-02,7.843006e-02


In [13]:
# Podemos observar que ya hay features numéricos con demasiados valores NULOS que podemos obviar de primeras
FEATURES2DROP = [
    "datos_residenciales_alcantarillado", 
    "datos_residenciales_fuente_agua", 
    "numero_unidad",
    "pies_cuadrados",
    "fecha_disponibilidad",
    "oferta_especial",
    "anio_construccion",
    ## Añado variables que no aportan importancia
    'es_destacado', 
    'avaluo_vivienda',
    'conteo_visitas_recorrido', 
    'oferta_inmediata_habilitada']
    
# Extraigo el precio de las variables FEATURES ya que es el target
TARGET = "precio"
FEATURES2DROP +=[TARGET] 


#### 3.2. Imputación de variables numericas

In [14]:
# Quitar los registros del train que tengan NaN
train_df = train_df[~train_df['precio'].isna()]

In [15]:
# Mediana por (baños, habitaciones)
median_map_full = train_df.groupby(["banos", "dormitorios"])["area_habitable"].median()

# Mediana solo por habitaciones
median_map_rooms = train_df.groupby(["dormitorios"])["area_habitable"].median()

# Mediana solo por baños
median_map_bath = train_df.groupby(["banos"])["area_habitable"].median()

# Mediana global del train
global_median = train_df["area_habitable"].median()

def impute_area(row):
    if not pd.isna(row["area_habitable"]):
        return row["area_habitable"]

    # 1. Por (baños, habitaciones)
    key_full = (row["banos"], row["dormitorios"])
    val = median_map_full.get(key_full)
    if not pd.isna(val):
        return val

    # 2. Solo por habitaciones
    val = median_map_rooms.get(row["dormitorios"])
    if not pd.isna(val):
        return val

    # 3. Solo por baños
    val = median_map_bath.get(row["banos"])
    if not pd.isna(val):
        return val

    # 4. Mediana global del train
    return global_median


def fill_nans_with_median(train_df, test_df, feature): 
    mediana_dorm = train_df[feature].median()
    train_df[feature] = train_df[feature].fillna(mediana_dorm)
    test_df[feature] = test_df[feature].fillna(mediana_dorm)
    return train_df, test_df

def fill_nans_with_mode(train_df, test_df, feature): 
    mediana_dorm = train_df[feature].mode()[0]
    train_df[feature] = train_df[feature].fillna(mediana_dorm)
    test_df[feature] = test_df[feature].fillna(mediana_dorm)
    return train_df, test_df


In [16]:
# Variables numéricas
# Imputar NaNs tanto en train como en test
train_df, test_df = fill_nans_with_mode(train_df, test_df, 'dormitorios')
train_df, test_df = fill_nans_with_mode(train_df, test_df, 'banos')
train_df, test_df = fill_nans_with_mode(train_df, test_df, 'dias_en_portal')

train_df["area_habitable"] = train_df.apply(impute_area, axis=1)
test_df["area_habitable"] = test_df.apply(impute_area, axis=1)

# Hay casos donde el value_counts es 1 o hay discrepancias en algunas casuísticas. 
# Por lo que decido seleccionar la zona horaria por ciudad con más frecuencia de aparición. 
zona_horaria_dominante = (train_df.groupby("codigo_postal")["zona_horaria"].agg(lambda x: x.value_counts().idxmax()))
train_df['zona_horaria'] = train_df['codigo_postal'].map(zona_horaria_dominante)
test_df['zona_horaria'] = test_df['codigo_postal'].map(zona_horaria_dominante)


## Tasa de impuesto de propiedad
# Ajustarla por codigo_postal a la mediana del valor del codigo postal

cp_impuesto_dict = dict(train_df.groupby('codigo_postal')['tasa_impuesto_propiedad'].median())
train_df['tasa_impuesto_propiedad'] = train_df['tasa_impuesto_propiedad'].fillna(train_df['codigo_postal'].map(cp_impuesto_dict))
# Si se mantiene algun NaN sustituir por la mediana global
train_df['tasa_impuesto_propiedad'] = train_df['tasa_impuesto_propiedad'].fillna(train_df['tasa_impuesto_propiedad'].median())

test_df['tasa_impuesto_propiedad'] = test_df['tasa_impuesto_propiedad'].fillna(test_df['codigo_postal'].map(cp_impuesto_dict))
test_df['tasa_impuesto_propiedad'] = test_df['tasa_impuesto_propiedad'].fillna(train_df['tasa_impuesto_propiedad'].median())


# Para latitud y longitud considerar la mediana por zona
media_lat_por_cp = dict(train_df.groupby('codigo_postal')['latitud'].mean())
media_long_por_cp = dict(train_df.groupby('codigo_postal')['longitud'].mean())

train_df['latitud'] = train_df['latitud'].fillna(train_df['codigo_postal'].map(media_lat_por_cp))
train_df['longitud'] = train_df['longitud'].fillna(train_df['codigo_postal'].map(media_long_por_cp))

test_df['latitud'] = test_df['latitud'].fillna(test_df['codigo_postal'].map(media_lat_por_cp))
test_df['longitud'] = test_df['longitud'].fillna(test_df['codigo_postal'].map(media_long_por_cp))

In [17]:
# Direccion no divulgada
# La gran mayoría de los casos son False, por lo que los NaNs los sustituiremos por False
train_df['direccion_no_divulgada'] = train_df['direccion_no_divulgada'].fillna(False)
test_df['direccion_no_divulgada'] = test_df['direccion_no_divulgada'].fillna(False)

## Es constructor premier
# La gran mayoria de los casos son False, por lo que los NaNs los sustituiremos por False
train_df['es_constructor_premier'] = train_df['es_constructor_premier'].fillna(False)
test_df['es_constructor_premier'] = test_df['es_constructor_premier'].fillna(False)

## Tiene video publico 
# Solo se informan los False por lo que entendemos que los NaN son True
train_df['tiene_video_publico'] = train_df['tiene_video_publico'].fillna(True)
test_df['tiene_video_publico'] = test_df['tiene_video_publico'].fillna(True)

## url_tour_virtual_terceros
# Rellenamos los Nan con el valor con mayor frecuencia --> False
train_df['url_tour_virtual_tercero_aprobado'] = train_df['url_tour_virtual_tercero_aprobado'].fillna(False)
test_df['url_tour_virtual_tercero_aprobado'] = test_df['url_tour_virtual_tercero_aprobado'].fillna(False)

In [18]:
# Hay un negativo en dias en portal --> transformo en valor absoluto 
train_df['dias_en_portal'] = train_df['dias_en_portal'].apply(lambda x: np.abs(x))
test_df['dias_en_portal'] = test_df['dias_en_portal'].apply(lambda x: np.abs(x))


In [19]:
def transformacion_log1p_variables_right_skewed_train_test(train_df, test_df, feature): 
    train_df[feature] = train_df[feature].apply(lambda x: np.log1p(x))
    test_df[feature] = test_df[feature].apply(lambda x: np.log1p(x))
    return (train_df, test_df)

# Como dias en portal es una variable right skewed sustituyo valores que superen el limite de outliers por la mediana 
train_df, test_df = transformacion_log1p_variables_right_skewed_train_test(train_df, test_df, 'dias_en_portal')
train_df, test_df = transformacion_log1p_variables_right_skewed_train_test(train_df, test_df, 'dormitorios')
train_df, test_df = transformacion_log1p_variables_right_skewed_train_test(train_df, test_df, 'banos')
train_df, test_df = transformacion_log1p_variables_right_skewed_train_test(train_df, test_df, 'area_habitable')

In [20]:
from sklearn.model_selection import train_test_split
# Usar autogluon 
from autogluon.tabular import TabularPredictor

In [21]:
train_df.columns

Index(['ciudad', 'estado', 'estado_propiedad', 'dormitorios', 'banos',
       'precio', 'anio_construccion', 'codigo_postal', 'longitud', 'latitud',
       ...
       'emb_246', 'emb_247', 'emb_248', 'emb_249', 'emb_250', 'emb_251',
       'emb_252', 'emb_253', 'emb_254', 'emb_255'],
      dtype='object', length=325)

In [22]:

X = train_df[[f for f in train_df[[f for f in train_df.columns if f not in FEATURES2DROP + ['descripcion_embeddings', 'codigo_postal', 'conteo fotos']]] if f not in ['precio']]]
y = train_df['precio'].apply(lambda x: np.log1p(x))
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# AutoGluon espera un DataFrame completo con la columna objetivo
train_data = X_train.copy()
train_data['target'] = y_train

test_data = X_test.copy()
test_data['target'] = y_test

In [23]:
# Definir el predictor
predictor = TabularPredictor(label='target', eval_metric='mae').fit(
    train_data=train_data,
    time_limit=2500,  # tiempo máximo en segundos
    presets='medium_quality'  # opcional: hace un entrenamiento más completo
)

No path specified. Models will be saved in: "AutogluonModels\ag-20251211_101431"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.10.19
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26100
CPU Count:          12
Memory Avail:       6.19 GB / 15.93 GB (38.9%)
Disk Space Avail:   78.84 GB / 222.73 GB (35.4%)
Presets specified: ['medium_quality']
Using hyperparameters preset: hyperparameters='default'
Beginning AutoGluon training ... Time limit = 2500s
AutoGluon will save models to "c:\Users\rlope\Documents\Master UB IA mercados financieros\repos\proyecto_hc\notebooks\AutogluonModels\ag-20251211_101431"
Train Data Rows:    87974
Train Data Columns: 313
Label Column:       target
AutoGluon infers your prediction problem is: 'regression' (because dtype of label-column == float and many unique label-values observed).
	Label info (max, min, mean, stddev): (19.467999741741696, 0.0

[1000]	valid_set's l1: 0.221747
[2000]	valid_set's l1: 0.214803
[3000]	valid_set's l1: 0.211373
[4000]	valid_set's l1: 0.208799
[5000]	valid_set's l1: 0.207333
[6000]	valid_set's l1: 0.206414
[7000]	valid_set's l1: 0.20545
[8000]	valid_set's l1: 0.204675
[9000]	valid_set's l1: 0.204365
[10000]	valid_set's l1: 0.203962


	-0.204	 = Validation score   (-mean_absolute_error)
	388.87s	 = Training   runtime
	0.49s	 = Validation runtime
Fitting model: LightGBM ... Training model for up to 1793.13s of the 1793.13s of remaining time.
	Fitting with cpus=6, gpus=0, mem=1.5/6.1 GB


[1000]	valid_set's l1: 0.219229
[2000]	valid_set's l1: 0.214046
[3000]	valid_set's l1: 0.21269
[4000]	valid_set's l1: 0.211712
[5000]	valid_set's l1: 0.210983
[6000]	valid_set's l1: 0.210419
[7000]	valid_set's l1: 0.210099
[8000]	valid_set's l1: 0.209919
[9000]	valid_set's l1: 0.209791
[10000]	valid_set's l1: 0.209708


	-0.2097	 = Validation score   (-mean_absolute_error)
	444.74s	 = Training   runtime
	0.65s	 = Validation runtime
Fitting model: RandomForestMSE ... Training model for up to 1346.98s of the 1346.98s of remaining time.
	Fitting with cpus=12, gpus=0, mem=0.1/6.0 GB
	-0.2739	 = Validation score   (-mean_absolute_error)
	570.57s	 = Training   runtime
	0.07s	 = Validation runtime
Fitting model: CatBoost ... Training model for up to 776.18s of the 776.18s of remaining time.
	Fitting with cpus=6, gpus=0, mem=2.5/5.7 GB
	Ran out of time, early stopping on iteration 2747.
	-0.2237	 = Validation score   (-mean_absolute_error)
	776.13s	 = Training   runtime
	0.12s	 = Validation runtime
Fitting model: WeightedEnsemble_L2 ... Training model for up to 360.00s of the -0.13s of remaining time.
	Ensemble Weights: {'LightGBMXT': 0.526, 'LightGBM': 0.368, 'CatBoost': 0.105}
	-0.1981	 = Validation score   (-mean_absolute_error)
	0.04s	 = Training   runtime
	0.0s	 = Validation runtime
AutoGluon training co

In [24]:
results = predictor.evaluate(test_data)
print(results)

{'mean_absolute_error': -0.1965640562214647, 'root_mean_squared_error': np.float64(-0.33317285432004773), 'mean_squared_error': -0.11100415085576774, 'r2': 0.929901836368973, 'pearsonr': 0.9645688418844147, 'median_absolute_error': -0.12919538292645516}


In [25]:
# codigo postal y conteo fotos molestan en el predict

In [26]:
y_test_predicted = predictor.predict(test_df[[f for f in train_df.columns if f not in FEATURES2DROP + ['descripcion_embeddings', 'codigo_postal', 'conteo fotos']]])

In [27]:
# Transformación inversa a escala original
final_y_pred = np.expm1(y_test_predicted)

In [28]:
len(y_test_predicted)

39867

In [29]:
len(y_test_predicted)

39867

In [30]:
submission_df = pd.DataFrame(test_ids_list)
submission_df

,0
0,110001
1,110002
2,110003
3,110004
4,110005
...,...
39862,149863
39863,149864
39864,149865
39865,149866


In [31]:
submission_df = pd.DataFrame(test_ids_list)
submission_df['precio'] = final_y_pred
submission_df = submission_df.rename(columns={0:'id'})
submission_df.head(3)

,id,precio
0,110001,1.061097e+06
1,110002,4.094994e+05
2,110003,1.546010e+05


In [32]:
submission_df.to_csv('../data/submission9.csv',index=False)

In [ ]:
test_df.shape

(39867, 738)